# SolarVoix — Analyse des résultats

**Notebook Fabric :** `NB_Analyse_Resultats`

**Lakehouse requis :** attachez `LH_SolarVoix` comme Lakehouse par défaut.

Ce notebook visualise les résultats validés dans les Ateliers 1 et 2. Il ne modifie aucune table.


## Cellule 1 — Chargement des données


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = spark.table("silver_appels_enrichis").toPandas()
df_historique = df[df["sentiment_reel"].notna()].copy()

df_historique["sentiment_reel_normalise"] = (
    df_historique["sentiment_reel"]
    .replace({"tres_negatif": "negatif"})
)

print(f"Appels Silver          : {len(df)}")
print(f"Appels avec vérité IA  : {len(df_historique)}")


## Cellule 2 — Volume et score par sentiment


In [ ]:
resume_sentiment = (
    df.groupby("sentiment_ia", dropna=False)
      .agg(
          nb_appels=("call_id", "count"),
          score_moyen=("score_risque_churn", "mean"),
          nb_alertes=("alerte_critique", "sum")
      )
      .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(
    data=resume_sentiment,
    x="sentiment_ia",
    y="nb_appels",
    hue="sentiment_ia",
    legend=False,
    ax=axes[0]
)
axes[0].set_title("Nombre d'appels par sentiment IA")
axes[0].set_xlabel("Sentiment IA")
axes[0].set_ylabel("Nombre d'appels")

sns.barplot(
    data=resume_sentiment,
    x="sentiment_ia",
    y="score_moyen",
    hue="sentiment_ia",
    legend=False,
    ax=axes[1]
)
axes[1].axhline(50, color="red", linestyle="--", label="Seuil critique = 50")
axes[1].set_title("Score churn moyen par sentiment")
axes[1].set_xlabel("Sentiment IA")
axes[1].set_ylabel("Score moyen")
axes[1].legend()

plt.tight_layout()
plt.show()


## Cellule 3 — Matrice de confusion normalisée


In [ ]:
matrice = pd.crosstab(
    df_historique["sentiment_reel_normalise"],
    df_historique["sentiment_ia"]
)

plt.figure(figsize=(6, 4))
sns.heatmap(matrice, annot=True, fmt="d", cmap="Blues")
plt.title("Sentiment réel normalisé vs sentiment IA")
plt.xlabel("Sentiment prédit")
plt.ylabel("Sentiment réel normalisé")
plt.tight_layout()
plt.show()


## Cellule 4 — Alertes critiques par produit


In [ ]:
alertes_par_produit = (
    df[df["alerte_critique"] == True]
    .assign(produit=lambda x: x["produit"].fillna("Métadonnée absente"))
    .groupby("produit")
    .size()
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 4))
alertes_par_produit.plot(kind="bar", color="#C0392B")
plt.title("Nombre d'alertes critiques par produit")
plt.xlabel("Produit")
plt.ylabel("Nombre d'alertes")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
